# Bước 3 — Train & so sánh ALS: baseline vs sentiment-adjusted

**Thay đổi so với bản cũ:**

1. **Lọc bỏ đơn hàng nhiều sản phẩm khác nhau** trước khi ghép review vào
   `order_items`. Bản cũ: 1 review theo `order_id` bị nhân bản cho MỌI
   `product_id` trong đơn, dù review chỉ nói về 1 món — gây nhiễu label. Bản
   mới: chỉ giữ đơn có đúng 1 `order_item_id`.
2. **1 lần split duy nhất** thành train/test (trên toàn bộ user-item-rating),
   thay vì split riêng cho baseline và sentiment rồi join lại — đơn giản hơn
   và đảm bảo 2 model được đánh giá trên cùng 1 tập test tuyệt đối.
3. **`adjusted_score` được tính với nhiều giá trị `ALPHA`** (grid-search
   0.1 → 0.9), KHÔNG cố định `ALPHA=0.4` như trước.
4. **Định nghĩa "relevant item" để tính Precision@10/NDCG@10 dựa trên
   `review_score` gốc** (`>= 4`), KHÔNG dựa trên `adjusted_score`. Lý do:
   nếu dùng `adjusted_score` để vừa tính label huấn luyện vừa định nghĩa
   "đúng/sai" khi đánh giá, phép đánh giá bị **circular** (tự chấm điểm cho
   chính mình) — model sentiment sẽ luôn có vẻ thắng dù không thực sự tốt
   hơn. Dùng `review_score` gốc làm ground-truth độc lập.
5. **Dùng `pyspark.mllib.evaluation.RankingMetrics`** (built-in, đã kiểm
   chứng) để tính Precision@10 và NDCG@10, thay vì tự viết UDF thủ công.
6. **Không so sánh RMSE chéo giữa 2 target khác thang đo.** RMSE chỉ dùng để
   model tự đánh giá với chính target của nó — kết luận "tốt hơn/kém hơn"
   dựa trên Precision@10/NDCG@10.
7. **Lưu model đầy đủ** bằng `.write().overwrite().save(...)` — bản cũ thiếu
   bước này.


In [1]:
import sys
sys.path.append(".")
from config import MODELS_DIR, MONGO_URI, MONGO_DB, get_model_path

print("MODELS_DIR =", MODELS_DIR)


MODELS_DIR = C:\Users\MY LAPTOP\PycharmProjects\Sentiment\models


In [2]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# Xem ghi chú trong 39_load_to_mongo.ipynb: cần connector 11.x cho Spark 4.x.
MONGO_CONNECTOR_PACKAGE = "org.mongodb.spark:mongo-spark-connector_2.13:11.1.0"

spark = (
    SparkSession.builder
    .appName("OlistALSTrain")
    .config("spark.jars.packages", MONGO_CONNECTOR_PACKAGE)
    .config("spark.mongodb.read.connection.uri", MONGO_URI)
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
spark


## Đọc dữ liệu từ MongoDB

In [3]:
def read_mongo(collection: str):
    return (
        spark.read.format("mongodb")
        .option("connection.uri", MONGO_URI)
        .option("database", MONGO_DB)
        .option("collection", collection)
        .load()
    )

orders = read_mongo("orders").select("order_id", "customer_id")
items = read_mongo("order_items").select("order_id", "order_item_id", "product_id")
reviews_sent = read_mongo("reviews_sentiment").select(
    "order_id", "review_score", "sentiment_score", "is_fake"
)

print("orders:", orders.count(), "| items:", items.count(), "| reviews_sentiment:", reviews_sent.count())


orders: 99441 | items: 112650 | reviews_sentiment: 40967


## Lọc đơn hàng nhiều sản phẩm khác nhau

Chỉ giữ đơn có đúng 1 `order_item_id` — tránh 1 review bị gán cho nhiều
product khác nhau trong cùng đơn (xem markdown đầu notebook).

In [4]:
items_per_order = items.groupBy("order_id").agg(F.count("order_item_id").alias("n_items"))
single_item_orders = items_per_order.filter(F.col("n_items") == 1).select("order_id")

n_before = items.select("order_id").distinct().count()
n_after = single_item_orders.count()
print(f"Số đơn hàng: {n_before:,} -> sau khi lọc chỉ giữ đơn 1 sản phẩm: {n_after:,} "
      f"({n_after / n_before:.1%})")

items_single = items.join(single_item_orders, on="order_id", how="inner")


Số đơn hàng: 98,666 -> sau khi lọc chỉ giữ đơn 1 sản phẩm: 88,863 (90.1%)


## Ghép dữ liệu: customer - product - review_score - sentiment_score

In [5]:
merged = (
    items_single
    .join(orders, on="order_id", how="inner")
    .join(reviews_sent, on="order_id", how="inner")
    .select("customer_id", "product_id", "review_score", "sentiment_score", "is_fake")
    .dropna(subset=["customer_id", "product_id", "review_score", "sentiment_score"])
)

print(f"Số dòng dữ liệu huấn luyện: {merged.count():,}")
merged.show(5)


Số dòng dữ liệu huấn luyện: 35,479
+--------------------+--------------------+------------+---------------+-------+
|         customer_id|          product_id|review_score|sentiment_score|is_fake|
+--------------------+--------------------+------------+---------------+-------+
|ae3f264a566530eba...|daa634b8673533acf...|           2|            1.0|   true|
|7bd970d498ab2a151...|11182511f85568220...|           5|            1.0|  false|
|8cf88d7ba142365ef...|e9a69340883a438c3...|           5|           0.75|  false|
|a87a24ee764f092e8...|bed48285a1d145ff8...|           2|            0.0|  false|
|8c89a09d8fb33b6e5...|d86a6c48f83b045cb...|           5|            1.0|  false|
+--------------------+--------------------+------------+---------------+-------+
only showing top 5 rows


In [6]:
from pyspark.ml.feature import StringIndexer

user_indexer = StringIndexer(inputCol="customer_id", outputCol="user_idx", handleInvalid="skip")
item_indexer = StringIndexer(inputCol="product_id", outputCol="item_idx", handleInvalid="skip")

merged_idx = user_indexer.fit(merged).transform(merged)
merged_idx = item_indexer.fit(merged_idx).transform(merged_idx)
merged_idx = merged_idx.select(
    "user_idx", "item_idx", "review_score", "sentiment_score", "is_fake"
).cache()

print(f"Số user: {merged_idx.select('user_idx').distinct().count():,}")
print(f"Số item: {merged_idx.select('item_idx').distinct().count():,}")


Số user: 35,363
Số item: 16,303


## Split train/test — 1 LẦN DUY NHẤT

Trước đây split riêng cho `train_raw/test_raw` và `train_adj/test_adj` (2 lần
`randomSplit` độc lập trên 2 DataFrame khác nhau) rồi mới join lại bằng
`user_idx, item_idx` để có `test_combined`. Cách đó chạy được nhưng vòng vo và
dễ nhầm. Ở đây split 1 lần trên `merged_idx` (đã có đủ `review_score` +
`sentiment_score` trên từng dòng) — mọi target/adjusted_score sau này đều tính
từ cùng 1 train/test set, đảm bảo tuyệt đối không lệch dòng.

In [7]:
train_df, test_df = merged_idx.randomSplit([0.8, 0.2], seed=42)
train_df = train_df.cache()
test_df = test_df.cache()

print(f"Train: {train_df.count():,} dòng | Test: {test_df.count():,} dòng")


Train: 28,268 dòng | Test: 7,211 dòng


## Ground-truth "relevant item" cho Precision@10 / NDCG@10

Dùng `review_score` GỐC (`>= 4`) — độc lập với `sentiment_score`/`adjusted_score`,
tránh đánh giá circular (xem markdown đầu notebook).

In [8]:
RELEVANCE_THRESHOLD = 4  # trên thang review_score gốc 1-5

relevant_test = (
    test_df.filter(F.col("review_score") >= RELEVANCE_THRESHOLD)
    .groupBy("user_idx")
    .agg(F.collect_set("item_idx").alias("relevant_items"))
)
print(f"Số user có ít nhất 1 item relevant trong test: {relevant_test.count():,}")


Số user có ít nhất 1 item relevant trong test: 4,939


## Hàm train ALS + đánh giá RMSE / Precision@10 / NDCG@10

Dùng `pyspark.mllib.evaluation.RankingMetrics` (built-in) thay vì tự viết UDF.

In [9]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.mllib.evaluation import RankingMetrics

K = 10

def train_als(train_df, label_col: str, rank=50, max_iter=10, reg_param=0.1):
    als = ALS(
        userCol="user_idx", itemCol="item_idx", ratingCol=label_col,
        rank=rank, maxIter=max_iter, regParam=reg_param,
        coldStartStrategy="drop", nonnegative=False,
    )
    return als.fit(train_df)


def evaluate_rmse(model, test_df, label_col: str) -> float:
    preds = model.transform(test_df)
    evaluator = RegressionEvaluator(metricName="rmse", labelCol=label_col, predictionCol="prediction")
    return evaluator.evaluate(preds)


def evaluate_ranking(model, test_df, relevant_df, k: int = K):
    """Precision@k và NDCG@k qua RankingMetrics (built-in, đáng tin cậy hơn UDF tự viết)."""
    users = test_df.select("user_idx").distinct()
    recs = model.recommendForUserSubset(users, k)
    recs_flat = recs.select(
        "user_idx",
        F.expr("transform(recommendations, x -> x.item_idx)").alias("rec_items"),
    )

    joined = (
        recs_flat.join(relevant_df, on="user_idx", how="inner")
        .rdd.map(lambda r: (r["rec_items"], list(r["relevant_items"])))
    )

    if joined.isEmpty():
        return None, None

    metrics = RankingMetrics(joined)
    return metrics.precisionAt(k), metrics.ndcgAt(k)


## Model baseline (label = `review_score` gốc)

In [10]:
model_baseline = train_als(train_df, label_col="review_score")

rmse_baseline = evaluate_rmse(model_baseline, test_df, label_col="review_score")
prec_baseline, ndcg_baseline = evaluate_ranking(model_baseline, test_df, relevant_test, k=K)

print(f"[baseline] RMSE (trên review_score) = {rmse_baseline:.4f}")
print(f"[baseline] Precision@{K} = {prec_baseline:.4f} | NDCG@{K} = {ndcg_baseline:.4f}")


C:\Users\MY LAPTOP\AppData\Local\Python\pythoncore-3.11-64\Lib\site-packages\pyspark\sql\context.py:157: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


[baseline] RMSE (trên review_score) = 1.5543
[baseline] Precision@10 = 0.0920 | NDCG@10 = 0.9200


## Grid-search `ALPHA` cho model sentiment

`adjusted_score = ALPHA * sentiment_score * 5 + (1 - ALPHA) * review_score`

Thử nhiều `ALPHA` thay vì cố định 0.4, chọn theo Precision@10/NDCG@10 (đánh
giá trên ground-truth độc lập ở cell trên) — không dùng RMSE để so sánh giữa
các ALPHA vì bản thân target `adjusted_score` thay đổi theo ALPHA nên RMSE
tuyệt đối giữa các ALPHA cũng không so sánh trực tiếp được; ranking metric
mới là căn cứ chọn ALPHA đúng.

In [11]:
def with_adjusted_score(df, alpha: float):
    return df.withColumn(
        "adjusted_score",
        F.col("sentiment_score") * 5 * F.lit(alpha) + F.col("review_score") * F.lit(1 - alpha),
    )

ALPHA_CANDIDATES = [0.1, 0.3, 0.5, 0.7, 0.9]
grid_results = []

for alpha in ALPHA_CANDIDATES:
    train_alpha = with_adjusted_score(train_df, alpha)
    test_alpha = with_adjusted_score(test_df, alpha)

    model_alpha = train_als(train_alpha, label_col="adjusted_score")
    prec, ndcg = evaluate_ranking(model_alpha, test_alpha, relevant_test, k=K)

    grid_results.append({"alpha": alpha, "precision_at_10": prec, "ndcg_at_10": ndcg})
    print(f"ALPHA={alpha:.1f} -> Precision@{K}={prec:.4f} | NDCG@{K}={ndcg:.4f}")

import pandas as pd
grid_df = pd.DataFrame(grid_results).sort_values("ndcg_at_10", ascending=False)
grid_df


ALPHA=0.1 -> Precision@10=0.0920 | NDCG@10=0.9200
ALPHA=0.3 -> Precision@10=0.0920 | NDCG@10=0.9200
ALPHA=0.5 -> Precision@10=0.0880 | NDCG@10=0.8800
ALPHA=0.7 -> Precision@10=0.0880 | NDCG@10=0.8254
ALPHA=0.9 -> Precision@10=0.0800 | NDCG@10=0.8000


,alpha,precision_at_10,ndcg_at_10
0,0.1,0.092,0.920000
1,0.3,0.092,0.920000
2,0.5,0.088,0.880000
3,0.7,0.088,0.825375
4,0.9,0.080,0.800000


In [12]:
BEST_ALPHA = float(grid_df.iloc[0]["alpha"])
print(f"ALPHA tốt nhất theo NDCG@{K}: {BEST_ALPHA}")

train_best = with_adjusted_score(train_df, BEST_ALPHA)
test_best = with_adjusted_score(test_df, BEST_ALPHA)

model_sentiment = train_als(train_best, label_col="adjusted_score")

rmse_sentiment = evaluate_rmse(model_sentiment, test_best, label_col="adjusted_score")
prec_sentiment, ndcg_sentiment = evaluate_ranking(model_sentiment, test_best, relevant_test, k=K)

print(f"[sentiment, ALPHA={BEST_ALPHA}] RMSE (trên adjusted_score) = {rmse_sentiment:.4f}")
print(f"[sentiment, ALPHA={BEST_ALPHA}] Precision@{K} = {prec_sentiment:.4f} | NDCG@{K} = {ndcg_sentiment:.4f}")


ALPHA tốt nhất theo NDCG@10: 0.1


C:\Users\MY LAPTOP\AppData\Local\Python\pythoncore-3.11-64\Lib\site-packages\pyspark\sql\context.py:157: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


[sentiment, ALPHA=0.1] RMSE (trên adjusted_score) = 1.5451
[sentiment, ALPHA=0.1] Precision@10 = 0.0920 | NDCG@10 = 0.9200


## Bảng so sánh cuối cùng

**Lưu ý quan trọng:** 2 dòng RMSE bên dưới đo trên 2 target khác nhau
(`review_score` vs `adjusted_score`) — **không so sánh trực tiếp % chênh lệch
giữa 2 RMSE này**. Căn cứ so sánh chính là Precision@10 / NDCG@10, vì cả 2
đều được đo trên cùng 1 tập test và cùng 1 định nghĩa "relevant" độc lập
(dựa trên `review_score` gốc).

In [13]:
comparison = pd.DataFrame([
    {"model": "baseline (review_score)", "rmse_on_own_target": rmse_baseline,
     "precision_at_10": prec_baseline, "ndcg_at_10": ndcg_baseline},
    {"model": f"sentiment (adjusted_score, ALPHA={BEST_ALPHA})", "rmse_on_own_target": rmse_sentiment,
     "precision_at_10": prec_sentiment, "ndcg_at_10": ndcg_sentiment},
])
comparison


,model,rmse_on_own_target,precision_at_10,ndcg_at_10
0,baseline (review_score),1.554306,0.092,0.92
1,"sentiment (adjusted_score, ALPHA=0.1)",1.545086,0.092,0.92


## Lưu model

Bổ sung so với bản cũ — bản cũ thiếu bước này.

In [14]:
baseline_path = str(get_model_path("model_baseline"))
sentiment_path = str(get_model_path("model_sentiment"))

model_baseline.write().overwrite().save(baseline_path)
model_sentiment.write().overwrite().save(sentiment_path)

print(f"Đã lưu model_baseline  -> {baseline_path}")
print(f"Đã lưu model_sentiment -> {sentiment_path} (ALPHA={BEST_ALPHA})")


Đã lưu model_baseline  -> C:\Users\MY LAPTOP\PycharmProjects\Sentiment\models\model_baseline
Đã lưu model_sentiment -> C:\Users\MY LAPTOP\PycharmProjects\Sentiment\models\model_sentiment (ALPHA=0.1)
